# Post5G - Simple Scenario

## Prerequisites

Before starting, make sure you:

- Have completed the [Preparation](./preparation.ipynb), [Provisioning Resources](./provision.ipynb), and [Kubernetes Cluster](./k8s.ipynb) tutorials.
- Have a basic understanding of Python.
- Are familiar with 5G architecture.

## Tutorial Overview

This scenario introduces you to the orchestrator by deploying a complete, fully virtualized 5G end-to-end network.

![Simple topology](figures/simple.png)

### What You Will Deploy

The network architecture consists of:
* **5G Core:** Implemented using **Open5GS**, deployed on a virtual machine (`node-core`).
* **gNB:** Implemented using **UERANSIM** (`gnb`), deployed on a virtual machine (`node-ran`).
* **User Equipment (UE):** Two simulated UEs (`sim_ue1` and `sim_ue2`) implemented via **UERANSIM**, deployed on a virtual machine (`node-ues`).

Because this network is fully virtualized with the core and RAN emulated and the radio channel/UEs simulated—there is no need for a physical radio unit or specific hardware. 

### UE Configuration

The network utilizes a single Data Network Name (DNN) (`internet`) and one Single-Network Slice Selection Assistance Information (S-NSSAI) (`internet`). The UEs are configured with the following parameters:

| UE | SUPI | S-NSSAI | DNN |
| :--- | :--- | :--- | :--- |
| `sim_ue1` | `001010000000100` | `internet` | `internet` |
| `sim_ue2` | `001010000000101` | `internet` | `internet` |

### Target Infrastructure

To run this scenario, we will use ephemeral virtual machines provided by the [SLICES Basic Infrastructure Service (SLICES BI)](https://doc.slices-ri.eu/BasicServices/BasicInfrastructure/basicinfrastructure.html). However, you can also adapt this deployment to use your own virtual machines or bare-metal servers as shown in the [Provisioning Resources tutorial](./provision.ipynb).

## Metamodel and Deployement model definition

### Metamodel
The Post-5G blueprint abstracts the 5G network with a generic **metamodel**
compliant with 3GPP specifications. The metamodel is essentially a
[Pydantic model](https://pydantic.dev/docs/validation/dev/get-started/) that
helps to define the settings of the 5G network to operate in a declarative way.

To simplify the usage of the metamodel, it comes with a full set of default
values. That way, the user only needs to specify the settings for elements that
deviate from the default values.

In this example we follow a leaning-by-doing approach, but you can visit the
[metamodel website](https://gitlab.inria.fr/slices-ri/blueprints/post-5g/meta-model)
to read the full specifications and default values.

In our case, we have to define a core, one gNB, and two UEs as stated above.

In this example, we will deploy a core with the default settings so we don't have
to explicitly mention and configure it.

We however have to declare the gNB and the UEs as described above. To that aim,
we simply have to instanciate an object of the `MetaModel` class and explicitely
populate it with our desired elements.

The following snippet defines the desired RAN and UEs.


In [1]:
from metamodel.models.metamodel import MetaModel

metamodel=MetaModel.from_yaml(
"""
ran:
    components:
        gnb:
            type: gnb
ues:
  sim_ue1:
    config:
      supi: '001010000000100'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - gnb
  sim_ue2:
    config:
      supi: '001010000000101'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - gnb
""")

**NOTE:** Because the radio channel is simulated, you must explicitly specify the gNBs to which the UEs connect.

### Understanding the Deployment Model

A key feature of the metamodel is that it is **implementation-agnostic**. This allows you to share 5G configurations and run experiments across various environments and implementations.

The orchestrator library instantiates these metamodels for specific environments by pairing them with a **deployment model**, which contains the configuration details required for a target deployment. The deployment model specifies:
* Which implementations to use for each 5G function.
* Where and how to deploy them.

Like the metamodel, the deployment model is a Pydantic model with built-in default values to simplify configuration. To create one, simply instantiate the `DeploymentModel` class and configure it to match your requirements.

### Configuring Functions and Target Nodes

By default, the core network uses Open5GS within our orchestrator, so no explicit configuration is required. However, you must explicitly declare that the gNB and UEs use **UERANSIM** by defining them under the `ran` and `ues` sections.

You also need to declare the target nodes for your functions. In the `nodes` section, you list these machines and tag them with the roles they support:
* **`node-core`:** Tagged with `core` and `ric` to run the core network and RIC.
* **`node-ran`:** Tagged with `ran` to accept RAN components.
* **`node-ues`:** Tagged with `ue` to host UE instances.

### Provisioning Resources with SLICES BI

To run this scenario using ephemeral virtual machines on the **SLICES BI** infrastructure, the orchestrator utilizes a collection of built-in provisioners (see the [Provisioning Resources](./provision.ipynb) tutorial). 

In this example, we use a provisioner named `my-provisionner` to automatically spawn and configure VMs, which is why each defined machine includes the `provisioner: my-provisionner` attribute.

Each provisioner accepts specific configuration parameters. For this tutorial:
* **Infrastructure:** `fr-sophia2-bi-vm1`
* **Project:** `post5g-beta`
* **Experiment Name:** `simple_scenario` (automatically created by the provisioner)
* **Node Lifetime:** 8 hours (after which they are automatically destroyed)

> **Important:** Make sure your public SSH keys are registered to SLICES before deploying your network. If you need a refresher on these concepts, check the [preparation tutorial](./preparation.ipynb).

> **Note:** Components are referenced by name in both the metamodel and deployment model. For example, `provisioner: my-provisionner` tells the orchestrator to use the provisioner explicitly named `my-provisionner`. This name-based referencing allows you to swap out provisioner types (e.g., switching from virtual machines to bare-metal servers or onboarded Linux hosts) without changing your core node definitions—making it ideal for exploring different parameter spaces.

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
ran:
  components:
    gnb:
      type: gnb
      implementation: ueransim
ues:
  sim_ue1:
    implementation: ueransim
    type: containerized
  sim_ue2:
    implementation: ueransim
    type: containerized
nodes:
  node-core:
    provisioner: my-provisionner
    roles:
    - core
    - ric
  node-ran:
    provisioner: my-provisionner
    roles:
    - ran
  node-ues:
    provisioner: my-provisionner
    roles:
    - ue
provisioners:
  my-provisionner:
    type: bi
    params:
      infra: fr-sophia2-bi-vm1
      project: post5g-beta
      experiment: simple_scenario
      create_experiment: true
      experiment_duration: 8h
      node_duration: 8h
"""
)



## Provisioning the Infrastructure

### Create Ephemeral Virtual Machines

Now that we have defined both the metamodel and deployment model, we can trigger resource provisioning using the appropriate **Manager**. 

Managers in the orchestrator are designed for specific operational tasks. The **`ProvisioningManagement`** manager handles provisioning the machines listed in the `nodes` section of the deployment model, automatically selecting the correct provisioner for each node.

1. **Initialize the Manager:** First, instantiate the `ProvisioningManagement` manager by passing in your metamodel and deployment model. This stage does not make changes to the infrastructure; it validates and reconciles the models to ensure consistency.
2. **Trigger Provisioning:** Once validated, execute the provisioning command to spawn and configure the actual target nodes.

In [3]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-core' provisioning] ***

TASK [provisioning/bi/provision : Select project 'post5g-beta' f

We now have our three virtual machines deployed in SLICES BI and configured according to the requirements of this scenario.

> **Optional:** You can verify the deployment by listing the experiment's active resources using the SLICES CLI (be sure to adapt the parameters to match your setup):

In [4]:
!SLICES_BI_SITE_ID=fr-sophia2-bi-vm1 slices bi list-resources --experiment simple_scenario

  Resources in experiment exp_expauth.ilabt.imec.be_01m5ax1q1qf628z85gxaf0g8vy  
┏━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┓
┃           ┃           ┃        ┃ Created   ┃ Expires  ┃ Public    ┃ Private  ┃
┃ ID        ┃ Name      ┃ Status ┃ At        ┃ At       ┃ IPv4      ┃ IPv4     ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━┩
│ r_fr-sop… │ node-core │ up     │ 2026-10-… │ 2026-10… │           │ 10.123.… │
│           │           │        │ 20:18 UTC │ 04:18    │           │          │
│           │           │        │           │ UTC      │           │          │
│ r_fr-sop… │ node-ran  │ up     │ 2026-10-… │ 2026-10… │           │ 10.123.… │
│           │           │        │ 20:18 UTC │ 04:18    │           │          │
│           │           │        │           │ UTC      │           │          │
│ r_fr-sop… │ node-ues  │ up     │ 2026-10-… │ 2026-10… │           │ 10.123.… │
│           │           │   

Each manager accepts a metamodel and a deployment model as input, and outputs an updated deployment model containing the results of its actions. This updated model can then be passed directly to the next manager in the workflow chain.

Several methods are available to inspect and extract these models:
* **`output()` Method:** Directly returns both the metamodel and the updated deployment model. 

As expected, the resulting deployment model includes all necessary access details (such as IP addresses and credentials) for the newly provisioned machines.

In [5]:
metamodel, deployment_model = prov.output()
print (deployment_model.to_yaml())

provisioners:
  my-provisionner:
    type: bi
    params:
      experiment: simple_scenario
      create_experiment: true
      experiment_duration: 8h
      node_duration: 8h
nodes:
  node-core:
    is_provisioned: true
    provisioner: my-provisionner
    ssh_info:
      hostname: 10.123.218.253
      user: dsaucezi
      jumphost:
        hostname: bastion1.theblueprintfactory.org
        user: proxy
        common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o
          ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o
          ControlPersist=7200s -o ControlPath=/tmp/cm-%C
      common_args: -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o
        ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s
        -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no
        -o UserKnownHostsFile=/dev/null  -W %h:%p proxy@bastion1.theblueprintfactory.org
        -p 22

It can also be useful to save these models to files—for instance, to resume your work later or share the configuration across different workflow steps (we demonstrate this pattern later in the tutorial).

In [6]:
from pathlib import Path
DIR="state/simple"
Path(DIR).mkdir(parents=True, exist_ok=True)

prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

### Build the Kubernetes Cluster

The orchestrator relies on Kubernetes to deploy most of its 5G network functions. The next step is to create a Kubernetes cluster using the virtual machines provisioned in the previous step.

To accomplish this, we use the **`K8SManagement`** manager. Its role is to deploy and configure a Kubernetes cluster tailored to our target 5G network layout. In this scenario, the cluster will include `node-core`, `node-ran`, and `node-ues`, each labeled so that workloads are scheduled onto the correct machines (for example, UEs will only be scheduled on `node-ues`).

#### Manager Chaining and Immutability

Because the orchestrator follows a declarative and functional paradigm:
* **Automatic Model Inheritance:** Managers consume and output metamodels and deployment models. When instantiating the Kubernetes manager, simply specify the preceding manager (e.g., `prov`), and it will automatically inherit the updated models containing the newly provisioned machine details.
* **Immutability:** Metamodels and deployment models are copied when passed between managers. Input models are **never mutated** in-place by a manager.

#### Cluster Deployment

Once the manager is instantiated, execute the cluster deployment command. This provisions the Kubernetes cluster across all specified nodes and installs a valid `kubeconfig` file on each machine to simplify subsequent management tasks.

> **Note:** To onboard an existing Kubernetes cluster rather than provisioning a new one, please refer to the [Orchestrator Library Documentation](https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator).

In [7]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)            # don't show output

At this stage, the infrastructure is fully ready to support our 5G network
deployment.

But before deploying the 5G network, let's archive the metamodel and deployment
model resulting from this deployment.

In [8]:
k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

## Deploy the 5G Network

### Deploy the 5G Core

Now that the underlying infrastructure is fully provisioned, the first step is to deploy the 5G core network.

Deploying the core is straightforward: instantiate a **`CoreManagement`** manager and execute the deployment. Because all 5G-specific settings and deployment details are already defined in your metamodel and deployment model, the manager has everything it needs. As with previous stages, simply pass in the output from the preceding manager (i.e., `k8s`) to inherit the models automatically.

Once deployed, the core not only instantiates all required 5G network functions but also automatically registers the authentication and subscriber information for the UEs described in your metamodel into the User Data Repository (UDR).

In [9]:
from orchestrator.deployment import CoreManagement
core = CoreManagement(preceding=k8s)

In [10]:
await core.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nssf' chart] *****

TASK [helm/prep

By default, the Open5GS core is deployed in the `open5gs-core` namespace within the Kubernetes cluster, with its network functions restricted to nodes tagged with the `core` role.

#### Verifying the Core Deployment

To verify that the core functions are running correctly, you can list the active pods in that namespace. 

The **`ProvisioningManagement`** manager provides a simple yet powerful `run_node_command` function to execute shell commands directly on nodes. For example, you can run `kubectl get pods` on the `node-core` machine to check that all pods are deployed as expected.

> **Note:** Because the orchestrator follows a strict functional and immutable paradigm, managers never modify existing models in place. To run commands against the latest infrastructure state, you must instantiate a new **`ProvisioningManagement`** manager and import the models from the core deployment manager you just executed. An older `ProvisioningManagement` instance will still hold a stale deployment model that lacks the latest updates.

In [11]:
prov = ProvisioningManagement(preceding=core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core", "-o", "wide"],
    nodes=["node-core"],
    become=True, quiet=True)))

{'node-core': 'NAME                            READY   STATUS    RESTARTS   AGE     IP               NODE        NOMINATED NODE   READINESS GATES\nopen5gs-amf-7b6dcfbfdf-v9nx2    1/1     Running   0          2m34s   10.244.134.7     node-core   <none>           <none>\nopen5gs-ausf-5778d7548b-l9kh7   1/1     Running   0          2m17s   10.244.134.8     node-core   <none>           <none>\nopen5gs-bsf-75cf9d7fb4-bl75p    1/1     Running   0          24s     10.244.134.14    node-core   <none>           <none>\nopen5gs-database-0              1/1     Running   0          3m30s   10.244.134.5     node-core   <none>           <none>\nopen5gs-nrf-65d98494d7-kldgt    1/1     Running   0          2m58s   10.244.134.6     node-core   <none>           <none>\nopen5gs-nssf-84cb8d64f7-snjbr   1/1     Running   0          42s     10.244.134.13    node-core   <none>           <none>\nopen5gs-pcf-6f444b66c5-97lrv    1/1     Running   0          98s     10.244.134.10    node-core   <none>           

In a very similar way, the logs of 5G functions can be retrieved directly with
the managers. For example, the logs of the AMF shows that it is well registered
to the NRF.

In [12]:
print(await core.get_logs(components=["amf"], quiet=True))

Open5GS daemon v2.8.0

10/19 20:33:05.286: [app] INFO: Configuration: '/opt/open5gs/etc/open5gs/config.yaml' (../lib/app/ogs-init.c:144)
10/19 20:33:05.287: [app] INFO: File Logging: '/opt/open5gs/var/log/open5gs/amf.log' (../lib/app/ogs-init.c:147)
10/19 20:33:05.287: [app] INFO: LOG-LEVEL: 'info' (../lib/app/ogs-init.c:150)
10/19 20:33:05.291: [sbi] INFO: Setup NF EndPoint(fqdn) [open5gs-nrf:0] (../lib/sbi/context.c:451)
10/19 20:33:05.291: [metrics] INFO: metrics_server() [http://10.244.134.7]:9090 (../lib/metrics/prometheus/context.c:392)
10/19 20:33:05.291: [sbi] INFO: NF Service [namf-comm] (../lib/sbi/context.c:2012)
10/19 20:33:05.292: [sbi] INFO: nghttp2_server() [http://10.244.134.7]:80 (../lib/sbi/nghttp2-server.c:454)
10/19 20:33:05.320: [amf] INFO: ngap_server() [10.244.134.7]:38412 (../src/amf/ngap-sctp.c:61)
10/19 20:33:05.320: [sctp] INFO: AMF initialize...done (../src/amf/app.c:33)
10/19 20:33:05.324: [sbi] INFO: [49a45d62-cbfc-41f1-866a-134d2ec0acc6] NF registered [He

### Deploy the 5G RAN

Now that the 5G core is fully functional, you can deploy the Radio Access Network (RAN) using the **`RANManagement`** manager. 

Instantiate a `RANManagement` manager, import the metamodel and deployment model from the preceding deployment stage (i.e., `core`), and then trigger the deployment by calling the manager's `deploy()` method.

In [13]:
from orchestrator.deployment import RANManagement
ran = RANManagement(preceding=core)

In [14]:
await ran.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-gnb' chart] *****
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy local 'ueransim-gnb' Helm chart directory for 'ueransim-gnb'] ***
changed: [node-core]

TASK [helm/prepare : Render 'ueransim-gnb' Helm Chart.yaml file from Jinja template] ***
changed: [node-core]

TASK [helm/prepare : Render templated Helm files for 'ueransim-gnb'] ***********
changed: [node-core] => (item=values.yaml)
changed: [node-core] => (item=config.yaml)

PLAY RECAP *********************************************************************
node-core         

In this scenario, the RAN is configured with a single UERANSIM gNB named `gnb`. By default, the UERANSIM RAN is deployed in the `ueransim-ran` namespace. Following deployment, you should see a single active pod scheduled on the `node-ran` node.

Just as with the core network, you can verify this deployment by instantiating a new **`ProvisioningManagement`** manager and running a `kubectl` command to list the pods in the RAN namespace.

In [15]:
prov = ProvisioningManagement(preceding=ran)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "ueransim-ran", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True)))

{'node-ran': 'NAME                            READY   STATUS    RESTARTS   AGE   IP              NODE       NOMINATED NODE   READINESS GATES\nueransim-gnb-5cdc64dc84-bthqv   1/1     Running   0          13s   10.244.86.178   node-ran   <none>           <none>'}


The output of the command confirms the proper deployment of the gNB on the
`node-ran` machine.

To confirm that the gNB is implemented with UERANSIM and is registered to the
core, we can analyse the logs of the gNB.

In [16]:
print(await ran.get_logs(components=["gnb"], quiet=True))

UERANSIM v3.3.0
[2026-10-19 20:37:20.250] [sctp] [info] Trying to establish SCTP connection... (10.244.134.7:38412)
[2026-10-19 20:37:20.266] [sctp] [info] SCTP connection established (10.244.134.7:38412)
[2026-10-19 20:37:20.266] [sctp] [debug] SCTP association setup ascId[3]
[2026-10-19 20:37:20.266] [ngap] [debug] Sending NG Setup Request
[2026-10-19 20:37:20.278] [ngap] [debug] NG Setup Response received
[2026-10-19 20:37:20.278] [ngap] [info] NG Setup procedure is successful



From the logs, we can confirm that the NG Setup procedure with the AMF was successful. Correlating these logs with the core pod listing shows that the AMF address used by the gNB correctly matches the AMF address of the core deployed earlier. 

Crucially, this IP address configuration is handled transparently by the orchestrator library, freeing you from manual networking overhead.

## Deploy the UEs

Now that both the 5G core network and RAN are fully functional, you can deploy the UEs defined in your metamodel.

To do so, instantiate a **`UEManagement`** manager, passing in the metamodel and deployment model from the previous stage (i.e., `ran`). Then, trigger the UE deployment by calling the manager's `deploy()` method.

When using UERANSIM, UEs are deployed into the `ueransim-ue` namespace by default. As in previous steps, you can run a `kubectl` command via a `ProvisioningManagement` manager to confirm that both UEs are running successfully.

In [17]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)

In [18]:
await ues.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue1' chart] ***

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue2' chart] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy local 'ueransim-containerized' Helm chart directory for 'ueransim-sim-ue1'] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://d

In [19]:
prov = ProvisioningManagement(preceding=ues)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "ueransim-ue", "-o", "wide"],
    nodes=["node-ues"],
    become=True, quiet=True)))

{'node-ues': 'NAME                                READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES\nueransim-sim-ue1-6db84b5878-h9rv6   1/1     Running   0          15s   10.244.198.2   node-ues   <none>           <none>\nueransim-sim-ue2-6485fd4bd6-6xgb8   1/1     Running   0          15s   10.244.198.3   node-ues   <none>           <none>'}


As for the gNB or the AMF we can also analyse the logs the the UEs and see that
they both successuflly registered to the network and established a PDU session.

In [20]:
print(await ues.get_logs(components=["sim_ue1"], quiet=True))

UERANSIM v3.3.0
[2026-10-19 20:39:32.286] [nas] [info] UE switches to state [MM-DEREGISTERED/PLMN-SEARCH]
[2026-10-19 20:39:32.287] [rrc] [debug] New signal detected for cell[1], total [1] cells in coverage
[2026-10-19 20:39:32.287] [nas] [info] Selected plmn[001/01]
[2026-10-19 20:39:32.287] [rrc] [info] Selected cell plmn[001/01] tac[1] category[SUITABLE]
[2026-10-19 20:39:32.287] [nas] [info] UE switches to state [MM-DEREGISTERED/PS]
[2026-10-19 20:39:32.287] [nas] [info] UE switches to state [MM-DEREGISTERED/NORMAL-SERVICE]
[2026-10-19 20:39:32.287] [nas] [debug] Initial registration required due to [MM-DEREG-NORMAL-SERVICE]
[2026-10-19 20:39:32.287] [nas] [debug] UAC access attempt is allowed for identity[0], category[MO_sig]
[2026-10-19 20:39:32.287] [nas] [debug] Sending Initial Registration
[2026-10-19 20:39:32.288] [nas] [info] UE switches to state [MM-REGISTER-INITIATED]
[2026-10-19 20:39:32.288] [rrc] [debug] Sending RRC Setup Request
[2026-10-19 20:39:32.288] [rrc] [info] R

In [21]:
print(await ues.get_logs(components=["sim_ue2"], quiet=True))

UERANSIM v3.3.0
[2026-10-19 20:39:32.879] [nas] [info] UE switches to state [MM-DEREGISTERED/PLMN-SEARCH]
[2026-10-19 20:39:32.880] [rrc] [debug] New signal detected for cell[1], total [1] cells in coverage
[2026-10-19 20:39:32.880] [nas] [info] Selected plmn[001/01]
[2026-10-19 20:39:32.880] [rrc] [info] Selected cell plmn[001/01] tac[1] category[SUITABLE]
[2026-10-19 20:39:32.880] [nas] [info] UE switches to state [MM-DEREGISTERED/PS]
[2026-10-19 20:39:32.880] [nas] [info] UE switches to state [MM-DEREGISTERED/NORMAL-SERVICE]
[2026-10-19 20:39:32.880] [nas] [debug] Initial registration required due to [MM-DEREG-NORMAL-SERVICE]
[2026-10-19 20:39:32.880] [nas] [debug] UAC access attempt is allowed for identity[0], category[MO_sig]
[2026-10-19 20:39:32.880] [nas] [debug] Sending Initial Registration
[2026-10-19 20:39:32.880] [nas] [info] UE switches to state [MM-REGISTER-INITIATED]
[2026-10-19 20:39:32.880] [rrc] [debug] Sending RRC Setup Request
[2026-10-19 20:39:32.881] [rrc] [info] R

To test that the UEs are properly attached and that the network functions as expected, you can send ICMP ping requests from the UEs over their 5G radio interface. When using UERANSIM, this virtual interface is named `uesimtun0`.

To run commands directly inside the UE containers on the node, use the `run_node_command` function provided by the `ProvisioningManagement` manager to execute a `kubectl exec` command against the target UE pod.

In [22]:
command = ["ping", "-I", "uesimtun0", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True)

ping from `sim_ue1`:

In [23]:
print (pings["sim_ue1"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 uesimtun0: 56(84) bytes of data.
[1792442410.251051] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=5.14 ms
[1792442411.252041] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=4.93 ms
[1792442412.254420] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=5.30 ms
[1792442413.255686] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=5.16 ms
[1792442414.256790] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=5.05 ms
[1792442415.257789] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=4.93 ms
[1792442416.259744] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=4.87 ms
[1792442417.261843] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=5.02 ms
[1792442418.262695] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=4.78 ms
[1792442419.277066] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=17.3 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9014ms
rtt min/avg/max/mdev = 4.784/6.245/17.279/3.680 ms



ping from `sim_ue2`:

In [24]:
print (pings["sim_ue2"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 uesimtun0: 56(84) bytes of data.
[1792442410.409960] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=4.58 ms
[1792442411.411879] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=4.87 ms
[1792442412.413944] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=4.97 ms
[1792442413.414841] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=4.83 ms
[1792442414.415768] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=4.84 ms
[1792442415.417682] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=4.86 ms
[1792442416.419707] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=4.97 ms
[1792442417.420639] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=4.87 ms
[1792442418.422897] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=5.18 ms
[1792442419.437958] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=19.0 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9014ms
rtt min/avg/max/mdev = 4.575/6.291/18.963/4.226 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [25]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")